# part 1: ---> Data Preprocessing

In [2]:
# Task 1: Load and Understand the Dataset

# Import Pandas
import pandas as pd

# Load Dataset
df = pd.read_csv("tmdb_5000_movies.csv")

# 1. Dataset Shape
print("Dataset Shape:")
print(df.shape)

# 2. Column Names
print("\nColumn Names:")
print(df.columns.tolist())

# 3. First 5 Rows
print("\nFirst 5 Rows:")
print(df.head())

# 4. Other Essential Details
print("\nDataset Information:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

# 5. Identify Text Column
print("\nText Column Used for Recommendation: overview")

Dataset Shape:
(4803, 20)

Column Names:
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']

First 5 Rows:
      budget                                             genres  \
0  237000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
1  300000000  [{"id": 12, "name": "Adventure"}, {"id": 14, "...   
2  245000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   
3  250000000  [{"id": 28, "name": "Action"}, {"id": 80, "nam...   
4  260000000  [{"id": 28, "name": "Action"}, {"id": 12, "nam...   

                                       homepage      id  \
0                   http://www.avatarmovie.com/   19995   
1  http://disney.go.com/disneypictures/pirates/     285   
2   http://www.sonypictures.com/movies/spectre/  206647   
3            http://www

In [ ]:
# rask 2: ---> Text preprocessing for recommendation 
import pandas as pd
import re
from nltk.corpus import stopwords
import nltk

# Download stopwords
nltk.download('stopwords')

# Load dataset
df = pd.read_csv("tmdb_5000_movies.csv")

# Select text column (overview)
df["overview"] = df["overview"].fillna("")

# English stopwords
stop_words = set(stopwords.words("english"))

# Text preprocessing function
def preprocess(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove stopwords
    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

# Apply preprocessing
df["clean_text"] = df["overview"].apply(preprocess)

# Show output
print(df[["overview", "clean_text"]].head())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                            overview  \
0  In the 22nd century, a paraplegic Marine is di...   
1  Captain Barbossa, long believed to be dead, ha...   
2  A cryptic message from Bond’s past sends him o...   
3  Following the death of District Attorney Harve...   
4  John Carter is a war-weary, former military ca...   

                                          clean_text  
0  nd century paraplegic marine dispatched moon p...  
1  captain barbossa long believed dead come back ...  
2  cryptic message bonds past sends trail uncover...  
3  following death district attorney harvey dent ...  
4  john carter warweary former military captain w...  


# part 2 --> Text vectorization

In [ ]:
# task 3: ---> Vectorization using TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

print("Shape of TF-IDF Matrix:", tfidf_matrix.shape)
print(vectorizer.get_feature_names_out()[:20])

Shape of TF-IDF Matrix: (4803, 5000)
['aaron' 'abandoned' 'abandons' 'abducted' 'abilities' 'ability' 'able'
 'aboard' 'abroad' 'absence' 'abuse' 'abused' 'abusive' 'academic'
 'academy' 'accept' 'accepted' 'accepts' 'access' 'accident']


In [ ]:
# task 4:---> similarity compuation
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

# Display results
print("Shape of Similarity Matrix:", similarity_matrix.shape)
print(similarity_matrix[:5, :5])

Shape of Similarity Matrix: (4803, 4803)
[[1.         0.         0.         0.03405426 0.02096406]
 [0.         1.         0.021662   0.         0.04063516]
 [0.         0.021662   1.         0.         0.        ]
 [0.03405426 0.         0.         1.         0.01515428]
 [0.02096406 0.04063516 0.         0.01515428 1.        ]]


# part 3: ---> Recommendation Logic

In [ ]:
# task 5:---> Recommendation function
def recommend(item_name, top_n=5):
    # Check if movie exists
    if item_name not in df['title'].values:
        return "Movie not found!"

    # Find movie index
    index = df[df['title'] == item_name].index[0]

    # Get similarity scores
    scores = list(enumerate(similarity_matrix[index]))

    # Sort by similarity score
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Remove the selected movie itself
    scores = scores[1:top_n+1]

    print(f"\nTop {top_n} recommendations for '{item_name}':\n")

    for i, score in scores:
        print(df.iloc[i]['title'], "-> Similarity:", round(score, 3))

In [5]:
recommend("Avatar")
recommend("Titanic")
recommend("the dark knight")



Top 5 recommendations for 'Avatar':

Apollo 18 -> Similarity: 0.228
The American -> Similarity: 0.194
The Inhabited Island -> Similarity: 0.159
Tears of the Sun -> Similarity: 0.157
The Matrix -> Similarity: 0.146

Top 5 recommendations for 'Titanic':

Amidst the Devil's Wings -> Similarity: 0.223
Ghost Ship -> Similarity: 0.191
The Switch -> Similarity: 0.17
WALL·E -> Similarity: 0.168
The Rose -> Similarity: 0.166


'Movie not found!'

In [6]:
print(df.columns)

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'clean_text'],
      dtype='str')
